# LAB05　while 循环

**对应课程：** 第 5 讲　程序的循环结构（while、break、continue、random）
**时长：** 60 分钟　　**环境：** Jupyter Lab　　**提交：** `学号_姓名_lab05.ipynb`

## 实验目标
1. 能判断什么时候用 while：**重复几次事先不知道，要看条件**
2. 会写 while 的三要素：**初值、条件、循环体里改变条件中的变量**
3. 能识别无限循环，并会在 Jupyter 中中止它
4. 会用 `break`、`continue`，以及 `while True` + `break` 的写法
5. 用 while 改写神经元训练：**训练到达标为止**

## 规则
- 写 while 之前先问自己：**循环体里哪个变量在变？它会不会让条件最终变成 False？**
- 程序卡住（单元格左边一直显示 `[*]`）：点工具栏的 ■，或菜单 **Kernel → Interrupt Kernel**。这在 while 实验里很正常，中止后改代码再运行即可。
- 每道编程题下方都有 `check(...)`：✅ 正确，❌ 不对，⬜ 还没做。
- **不要用 `input()`**。

## 评分标准
| 项目 | 分值 |
|---|---|
| 第 1 部分：预测与找 bug 填写完整 | 20 |
| 第 2 部分：3 道题 check 全部 ✅ | 30 |
| 第 3 部分：check 全部 ✅，3.4 能正确运行 | 20 |
| 第 4 部分：4.1、4.2 check ✅，4.3 表格填写完整 | 20 |
| 思考题 | 10 |
| 第 5 部分挑战 | 加分 5 |

In [1]:
# ===== 第 0 步：先运行这个单元格（选中后按 Shift + Enter）=====
import hashlib, math, random, statistics

def _norm(v):
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, (int, float)):
        return f"{float(v):.3f}"
    if isinstance(v, (list, tuple)):
        return "[" + ",".join(_norm(x) for x in v) + "]"
    return str(v)

_ANS = {
    "2.1 半衰期 6 小时：需要几小时": "35038d854f",
    "2.1 每小时代谢 15%：需要几小时": "81751cdaf4",
    "2.2 输完用时（分钟）": "bae389c7e4",
    "2.3 需要几个月": "04b57fbbc7",
    "3.1 第一个超标日": "4f82acd78f",
    "3.2 有效天数的平均值": "104086e2ff",
    "3.2 有效天数": "53b36e7bba",
    "3.3 一共读了几次": "4f82acd78f",
    "4.1 训练轮数": "2f85f5eff9",
    "4.2 学习率 0.0001 时的轮数": "a6c7e18143",
    "5.1 正常病人训练轮数": "dc120078a8",
    "5.2 给药次数": "954df141c8",
}

def check(name, value):
    """把你的结果和参考答案对照。它只告诉你对不对，不会把答案告诉你。"""
    if value is None:
        print(f"⬜ {name}：还没完成")
        return
    code = hashlib.md5(_norm(value).encode("utf-8")).hexdigest()[:10]
    if code == _ANS.get(name):
        print(f"✅ {name}：{value!r}  正确")
    else:
        print(f"❌ {name}：你的结果是 {value!r}，和参考答案不一致，再检查一下")

print("准备完成，可以开始了！")

准备完成，可以开始了！


## 遇到问题先查这里

| 现象 | 常见原因 | 怎么改 |
|---|---|---|
| 单元格左边一直是 `[*]`，程序停不下来 | **无限循环**：循环体里没有改变条件中的变量，或改错了方向 | 先点工具栏 ■（或 Kernel → Interrupt Kernel）中止；再检查“哪个变量在变、会不会让条件变成 False” |
| 屏幕被同一行输出刷满 | 无限循环里有 `print` | 同上；中止后可以点右键 → Clear Outputs 清空输出 |
| `NameError: name 'y' is not defined` | while 条件里用到的变量，循环前没有给初值 | 在 while 前面先写 `y = 0` 这样的初值 |
| 结果差了一轮 | `>=` 和 `>` 用混；计数器加在了错误的位置 | 画追踪表，一轮一轮核对 |
| 结果差得很远 | 计数器或累加器的初值写进了循环里 | 初值放在循环**前面** |
| `IndexError` | while 用下标遍历列表时没限制 `i < len(...)` | 条件写成 `while i < len(glucose):` |

## 第 1 部分　先预测，再运行（10 分钟）

### 1.1　倒计时
| 问题 | 我的预测 | 运行结果 |
|---|---|---|
| 输出是什么（逐行写） |3 2 1 出发！ 循环结束后 n=0|3 2 1 出发！ 循环结束后 n=0 |
| 循环体一共执行了几次 | 3|3 |
| 循环结束后 n 是几 | 0|0 |

In [3]:
n = 3
while n > 0:
    print(n)
    n = n - 1
print("出发！")
print("循环结束后 n =", n)

3
2
1
出发！
循环结束后 n = 0


### 1.2　追踪表：血药浓度何时降到安全线以下
某药初始浓度 80 mg/L，半衰期 6 小时。先填表，再运行。

| 轮次 | 进入前检查 `c >= 10`？ | c（这一轮执行后） | t（这一轮执行后） |
|---|---|---|---|
| 1 | 80 ≥ 10，成立 |40 | 6|
| 2 |40 ≥ 10,成立 |20 |12 |
| 3 | 20 ≥ 10，成立|10 | 18|
| 4 | 10 ≥ 10，成立|5 |24 |
| 5 | 5≤ 10 不成立|不执行|

最后一行输出，我的预测：24 5.0

**追问：** 把 `>=` 改成 `>`，输出会变成什么？先预测，再改代码验证。
    答 会少一次执行 c截止到10 t截止到18

In [3]:
c, t = 80, 0
while c >= 10:
    c = c / 2
    t = t + 6
print(t, c)

24 5.0


In [4]:
c, t = 80, 0
while c > 10:
    c = c / 2
    t = t + 6
print(t, c)

18 10.0


### 1.3　找 bug：哪几段永远停不下来？
**只看，不要运行！** 在表格里判断。

```python
# A
i = 0
while i < 5:
    print(i)
```
```python
# B
i = 0
while i < 5:
    i = i + 1
```
```python
# C
i = 10
while i > 0:
    i = i + 1
```
```python
# D
i = 1
while i != 10:
    i = i + 2
```

| 代码 | 会停 / 停不下来 | 为什么 |
|---|---|---|
| A |不停 |没有对i进行处理 作为0会一直小于5 |
| B | 会停|运行5次i就等于5了就会跳出循环了 |
| C |不会停 |一开始就大于条件 循环还是+1 只会越来越大 |
| D |不会停 |永远是奇数不会等于10 |

### 1.4　把 for 改写成 while
下面的 for 循环输出 1～5。在下一个单元格用 **while** 写出完全相同的输出。

In [4]:
for i in range(1, 6):
    print(i)

1
2
3
4
5


In [7]:
# ↓↓↓ 用 while 改写 ↓↓↓
i=0
while i!=5:
    i+=1
    print(i)

    

1
2
3
4
5


## 第 2 部分　while 基础（17 分钟）

### 2.1　药物代谢：多久降到安全线以下？
**(a)** 某药初始血药浓度 120 mg/L，半衰期 6 小时，安全线 8 mg/L。浓度降到安全线以下需要几小时？存入 `hours`。

**(b)** 换一种药：每小时被代谢掉 15%（即剩下 85%：`c = c * 0.85`），初始浓度和安全线不变。需要几小时？存入 `hours2`。

In [3]:
hours = None
hours2 = None
# ↓↓↓ (a) ↓↓↓
c,t=120,0
while c>=8:
    c/=2
    t+=6
hours=t
###b如下
c,t=120,0
while c>=8:
    c=c*0.85
    t+=1
hours2=t
# ↑↑↑
check("2.1 半衰期 6 小时：需要几小时", hours)
check("2.1 每小时代谢 15%：需要几小时", hours2)

✅ 2.1 半衰期 6 小时：需要几小时：24  正确
✅ 2.1 每小时代谢 15%：需要几小时：17  正确


### 2.2　输液：一分钟一分钟地模拟
500 mL 生理盐水，滴速 40 滴/分，滴系数 20 滴/mL，所以每分钟输入 2 mL。用 while **一分钟一分钟**地模拟，直到输完：
1. 每过 30 分钟打印一次剩余量（提示：`if t % 30 == 0:`）
2. 输完共用多少分钟，存入 `t`；再换算成“几小时几分钟”打印出来

期望输出（前两行和最后一行）：
```
第 30 分钟：剩余 440 mL
第 60 分钟：剩余 380 mL
……
共用 250 分钟，即 4 小时 10 分钟
```

In [2]:
t = None
# ↓↓↓ 在这里写代码 ↓↓↓
if t is None:
    t=0
c=500
while c!=0:
    c-=2
    t+=1
    if t % 30 == 0:
        print("第",t,"分钟：剩余",c,"ml")
print("共用",t,"分钟，即",t//60,"小时",t%60,"分钟")

# ↑↑↑
check("2.2 输完用时（分钟）", t)

第 30 分钟：剩余 440 ml
第 60 分钟：剩余 380 ml
第 90 分钟：剩余 320 ml
第 120 分钟：剩余 260 ml
第 150 分钟：剩余 200 ml
第 180 分钟：剩余 140 ml
第 210 分钟：剩余 80 ml
第 240 分钟：剩余 20 ml
共用 250 分钟，即 4 小时 10 分钟
✅ 2.2 输完用时（分钟）：250  正确


### 2.3　体重管理：几个月后 BMI 回到正常？
一位患者身高 1.70 m，体重 85 kg（BMI ≈ 29.4，按中国标准属于肥胖）。每月体重减少 2%（`weight = weight * 0.98`）。
1. 多少个月后 BMI **低于 24**？存入 `months`
2. 循环里打印每个月的体重和 BMI（保留 1 位小数）

In [4]:
months = 0#我把这里改了 None太繁琐了
# ↓↓↓ 在这里写代码 ↓↓↓
h,w,t=1.70,85,0
bmi=w/h**2
while bmi>=24:
    w*=0.98
    t+=1
    bmi=w/h**2
    print(t,"月体重：",w,"bmi:",round(bmi,1))
    
months=t


# ↑↑↑
check("2.3 需要几个月", months)

1 月体重： 83.3 bmi: 28.8
2 月体重： 81.634 bmi: 28.2
3 月体重： 80.00131999999999 bmi: 27.7
4 月体重： 78.40129359999999 bmi: 27.1
5 月体重： 76.83326772799998 bmi: 26.6
6 月体重： 75.29660237343998 bmi: 26.1
7 月体重： 73.79067032597118 bmi: 25.5
8 月体重： 72.31485691945176 bmi: 25.0
9 月体重： 70.86855978106271 bmi: 24.5
10 月体重： 69.45118858544146 bmi: 24.0
11 月体重： 68.06216481373262 bmi: 23.6
✅ 2.3 需要几个月：11  正确


## 第 3 部分　break、continue、while True（12 分钟）

先运行这个单元格，准备两周血糖数据。

In [ ]:
glucose = [5.6, 6.1, 5.9, 7.2, 6.0, 5.8, 6.3,
           7.5, 6.4, 5.7, 6.8, 7.1, 6.2, 5.9]

### 3.1　break：用 while 找第一个超标日
用 **while** 和下标 `i` 从第一天开始找，找到第一天 **≥ 7.0** 就 `break`。结果（第几天）存入 `first_high`。
提示：条件写成 `while i < len(glucose):`，防止越界；别忘了在循环体里让 `i` 变化。

In [3]:
first_high = 0#依旧
# ↓↓↓ 在这里写代码 ↓↓↓
glucose = [5.6, 6.1, 5.9, 7.2, 6.0, 5.8, 6.3,
           7.5, 6.4, 5.7, 6.8, 7.1, 6.2, 5.9]
i=0
while i<len(glucose):
    if glucose[i]>=7.0:
        first_high=i+1
        break
    i+=1
    
# ↑↑↑
check("3.1 第一个超标日", first_high)

✅ 3.1 第一个超标日：4  正确


### 3.2　continue：跳过漏测的日子
一周多记录里有几天漏测，记为 0。用 for + `continue` 跳过这些 0，求**实际测量日**的平均血糖 `avg_valid` 和有效天数 `n_valid`。

In [35]:
data = [5.6, 0, 6.1, 5.9, 0, 7.2, 6.0, 0, 5.8, 6.3]
avg_valid = 0
n_valid = 0
# ↓↓↓ 在这里写代码 ↓↓↓
for i in data:
    if i==0:
        continue
    avg_valid+=i
    n_valid+=1
avg_valid = avg_valid / n_valid 



    



    

# ↑↑↑
check("3.2 有效天数的平均值", avg_valid)
check("3.2 有效天数", n_valid)

✅ 3.2 有效天数的平均值：6.128571428571428  正确
✅ 3.2 有效天数：7  正确


### 3.3　while True + break：监护仪报警
用 random 模拟一台体温监护仪：每次读数是 36.0～39.5 之间的随机数（`random.uniform(36.0, 39.5)`）。
一直读下去，直到第一次读到 **超过 38.5℃** 就报警并停止。
1. 每次读数都打印出来（保留 1 位小数）
2. 一共读了几次，存入 `n_read`

要求：用 `while True:` + `break`；第一行先写 `random.seed(2026)`，保证全班结果一致。

In [6]:
n_read = 0#依旧
# ↓↓↓ 在这里写代码 ↓↓↓
random.seed(2026)
n=0
while True:
    i=random.uniform(36.0,39.5)
    n+=1
    if i>38.5:
        break
n_read=n
# ↑↑↑
check("3.3 一共读了几次", n_read)

✅ 3.3 一共读了几次：4  正确


### 3.4　for … else（课上的自学内容）
两周血糖里，**有没有**哪一天 ≥ 8.0？用 for … else 写：
- 找到了：打印“第 X 天超过 8.0”，然后 `break`
- 一直没找到（循环正常走完）：在 `else` 里打印“两周全部低于 8.0”

写完后把 8.0 改成 7.0 再运行一次，对比输出。

In [11]:
limit = 8.0
# ↓↓↓ 在这里写代码 ↓↓↓
glucose = [5.6, 6.1, 5.9, 7.2, 6.0, 5.8, 6.3,
           7.5, 6.4, 5.7, 6.8, 7.1, 6.2, 5.9]
for i in range(len(glucose)):
    if glucose[i]>=7.0:
        print("第",i+1,"天超过7.0")
        break
else:
    print("两周全部低于7.0")

第 4 天超过7.0


## 第 4 部分　综合：训练神经元，直到达标（15 分钟）

下面是第 5 讲的“神经元学习”程序，固定训练 50 轮。先运行它，看看 50 轮后的预测。

In [12]:
random.seed(7)
temp, hr = 39.0, 110          # 病人实测：体温 39℃，心率 110 次/分
x1 = temp - 37.3              # 体温比发热线 37.3℃ 高几度      → 1.7
x2 = (hr - 100) / 10          # 心率比 100 次/分 快几个“10 次” → 1.0
target = 1                    # 医生的结论：需要检查
w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)

for step in range(50):
    z = w1 * x1 + w2 * x2
    y = 1 / (1 + math.exp(-z))
    w1 = w1 + 0.1 * (target - y) * x1
    w2 = w2 + 0.1 * (target - y) * x2

print("50 轮后的预测：", round(y, 3))


50 轮后的预测： 0.94


问题：50 轮够吗？少一点行不行？——**事先不知道要训练几轮**，正是 while 的用武之地。

### 4.1　改成 while：训练到预测 y ≥ 0.95 为止
- 把 `for step in range(50):` 改成 while，条件是“预测还没到 0.95”
- 用计数器 `rounds` 记录训练了几轮
- 每一轮的顺序保持不变：**先算 z、y，再更新 w1、w2，最后 rounds 加 1**
- 注意：while 的条件里用到了 `y`，所以循环**前**必须先给 `y` 一个初值（比如 0）

In [15]:
random.seed(7)
w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)
rounds =0#依旧
# ↓↓↓ 在这里写代码 ↓↓↓
y=0
while y<=0.95:
    
    z = w1 * x1 + w2 * x2
    y = 1 / (1 + math.exp(-z)) 
    w1 = w1 + 0.1 * (target - y) * x1
    w2 = w2 + 0.1 * (target - y) * x2
    rounds+=1
    

# ↑↑↑
check("4.1 训练轮数", rounds)

✅ 4.1 训练轮数：59  正确


### 4.2　加一个“安全阀”
如果学习率很小，要训练非常多轮；如果程序有 bug，y 永远到不了 0.95，就成了无限循环。真实的训练程序都会设一个**轮数上限**。
1. 把学习率写成变量 `lr`，设为 `0.0001`
2. 把条件改成 `while y < 0.95 and rounds < 1000:`
3. 循环结束后用 if 判断：是“学会了”还是“到上限被迫停下”，打印出来

In [22]:
lr = 0.0001
random.seed(7)
w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)
rounds = 0#依旧
# ↓↓↓ 在这里写代码 ↓↓↓
y=0
while y < 0.95 and rounds < 1000:
    z = w1 * x1 + w2 * x2
    y = 1 / (1 + math.exp(-z)) 
    w1 = w1 + lr * (target - y) * x1
    w2 = w2 + lr* (target - y) * x2
    rounds+=1
if rounds<1000:
    print("学会了")
else:
    print("到上限被迫停下来")
          
# ↑↑↑
check("4.2 学习率 0.0001 时的轮数", rounds)

到上限被迫停下来
✅ 4.2 学习率 0.0001 时的轮数：1000  正确


In [29]:
lr = 0.0001

random.seed(7)
w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)
rounds = 0#依旧
# ↓↓↓ 在这里写代码 ↓↓↓
y=0
while y < 0.95 :
    z = w1 * x1 + w2 * x2
    y = 1 / (1 + math.exp(-z)) 
    w1 = w1 + lr * (target - y) * x1
    w2 = w2 + lr* (target - y) * x2
    rounds+=1

print(rounds)          

59045


### 4.3　学习率实验
把 4.2 的 `lr` 依次改成下表的值并运行（每次都会重新 `random.seed(7)`、重新初始化权重），记录轮数。

| 学习率 lr | 训练轮数 | 1000 轮内学会了吗 |
|---|---|---|
| 0.0001 |59045 |没有 |
| 0.05 | 118|学会了 |
| 0.1 |59 |学会了 |
| 0.5 |12 | 学会了|
| 1.0 |6 |学会了 |

**问题：** 学习率越大越好吗？写下你的看法。

我的看法：是 学习率越大 学得越快

## 第 5 部分　挑战（选做）

### 5.1　正常病人对照
换一位正常病人：体温 36.5℃、心率 75 次/分，医生结论“不用检查”（`target = 0`）。
用同样的方法训练（学习率 0.1，`random.seed(7)` 初始化权重），直到预测 **y ≤ 0.05**。轮数存入 `rounds_normal`，并打印最终的 w1、w2。

想一想：
- 这位病人的 x1、x2 是正还是负？
- 训练后 w1、w2 是变大还是变小？为什么目标是 0、x 是负数时，权重反而**变大**？

In [38]:
rounds_normal = None
# ↓↓↓ 在这里写代码 ↓↓↓
x1 = (36.5 - 37.3) / 1
x2 = (75 - 100) / 10
target = 0
lr = 0.1
rounds_normal = 0
y = 1.0

random.seed(7)
w1 = random.uniform(-1, 1)
w2 = random.uniform(-1, 1)

while y > 0.05:
    z = w1 * x1 + w2 * x2
    y = 1 / (1 + math.exp(-z))
    w1 = w1 + lr * (target - y) * x1
    w2 = w2 + lr * (target - y) * x2
    rounds_normal += 1


# ↑↑↑
check("5.1 正常病人训练轮数", rounds_normal)

✅ 5.1 正常病人训练轮数：35  正确


### 5.2　多次给药，浓度何时稳定？
某抗生素每 8 小时给药一次，每次使血药浓度增加 100 mg/L；半衰期 4 小时，所以 8 小时后只剩 1/4。
每次给药后的浓度：`c = c * 0.25 + 100`（第一次给药前 c = 0）。

用 while 模拟，直到**相邻两次给药后的浓度相差不到 1 mg/L**，认为达到“稳态”。求给了几次药，存入 `doses`，并打印每次给药后的浓度。

提示：循环里先记住旧值 `old = c`，再算新值，比较 `abs(c - old)`。用 `while True` + `break` 最方便。

In [37]:
doses = None
# ↓↓↓ 在这里写代码 ↓↓↓

doses = 0
c = 0

while True:
    old = c
    c = c * 0.25 + 100
    doses += 1
    print(f"第 {doses} 次给药后浓度：{c:.2f} mg/L")
    if abs(c - old) < 1:
        break
# ↑↑↑
check("5.2 给药次数", doses)

第 1 次给药后浓度：100.00 mg/L
第 2 次给药后浓度：125.00 mg/L
第 3 次给药后浓度：131.25 mg/L
第 4 次给药后浓度：132.81 mg/L
第 5 次给药后浓度：133.20 mg/L
✅ 5.2 给药次数：5  正确


## 思考题（课后完成，写在下面的单元格里）
1. 本次实验中，哪些题用 for 更合适、哪些用 while 更合适？各举一例，说出判断依据。
2. 1.3 中 D 为什么停不下来？只改一个符号，怎样让它停下？
3. 4.1 中，如果忘了在循环前写 `y = 0`，会报什么错？如果写成 `y = 1` 呢？
4. 3.2 中，把 `continue` 换成 `break`，结果会怎样？为什么？
5. 4.3 中，学习率越大越好吗？

**我的回答：**

1.知道次数的用for（比如说是个列表能数出来有几个或者能用len统计出来） 知道条件的用while（比如说数量数不过来 或者随机出数）

2.初始量是1 每一个循环＋2 则永远是奇数 不可能等于一个偶数10

3.会报错找不到变量y 主要是因为while中提到了y 如果等于1则下面都要再改直接就大于等于0.95了后面应该是在一直削减y

4.会直接结束 统计数据不全

5. 我觉得是 学习率大 学的快

## 提交前检查

- [ ] **Kernel → Restart Kernel and Run All Cells**，从头到尾没有报错
- [ ] 所有 `check` 都是 ✅（挑战题可以是 ⬜）
- [ ] 第 1 部分的预测表都填写了，预测错的写了原因
- [ ] 思考题已作答
- [ ] 保留运行结果（不要清除输出），保存后提交：`学号_姓名_lab05.ipynb`